# NLP




## 0) Install + imports

In [52]:
!pip -q install transformers datasets scikit-learn pandas numpy accelerate

In [53]:
import os, re, random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

TSV_PATH = "dontpatronizeme_pcl.tsv"
TRAIN_SPLIT_CSV = "train_semeval_parids-labels.csv"
DEV_SPLIT_CSV = "dev_semeval_parids-labels.csv"
TEST_TSV_PATH = "task4_test.tsv"

## 1) Load TSV + apply official split

In [54]:
df = pd.read_csv(
    TSV_PATH,
    sep="\t",
    skiprows=4,
    header=None,
    names=["par_id","art_id","keyword","country","text","label"]
)

df["label"] = df["label"].astype(int)
df["binary"] = (df["label"] >= 2).astype(int)

train_ids = pd.read_csv(TRAIN_SPLIT_CSV)["par_id"].astype(int)
dev_ids = pd.read_csv(DEV_SPLIT_CSV)["par_id"].astype(int)

train_df = df[df["par_id"].isin(train_ids)].copy()
dev_df   = df[df["par_id"].isin(dev_ids)].copy()

train_df["text"] = train_df["text"].fillna("").astype(str)
dev_df["text"]   = dev_df["text"].fillna("").astype(str)

train_df["keyword"] = train_df["keyword"].fillna("").astype(str)
dev_df["keyword"]   = dev_df["keyword"].fillna("").astype(str)

train_df.head()


,par_id,art_id,keyword,country,text,label,binary
0,1,@@24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0,0
1,2,@@21968160,migrant,gh,"In Libya today , there are countless number of...",0,0
2,3,@@16584954,immigrant,ie,White House press secretary Sean Spicer said t...,0,0
3,4,@@7811231,disabled,nz,Council customers only signs would be displaye...,0,0
4,5,@@1494111,refugee,ca,""" Just like we received migrants fleeing El Sa...",0,0


## 2) Internal split for tuning

In [55]:
from sklearn.model_selection import train_test_split

train_df["stratum"] = train_df["keyword"].astype(str) + "_" + train_df["binary"].astype(str)

train_tr_df, train_val_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["stratum"]
)

print("Train-tr:", len(train_tr_df), "Train-val:", len(train_val_df))


Train-tr: 7118 Train-val: 1257


## 3) Model + tokenizer

In [56]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

MAX_LEN = 256
MASK_TOKEN = tokenizer.mask_token or "<mask>"
print("Model:", MODEL_NAME)
print("Mask token:", MASK_TOKEN)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3600.61it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Model: roberta-base
Mask token: <mask>


## 4) Optional keyword masking augmentation

In [57]:
AUGMENT_KEYWORD_MASKING = True
MASK_PROB = 0.15

def maybe_mask_keyword(text: str, keyword: str, mask_token: str, p: float) -> str:
    if (not AUGMENT_KEYWORD_MASKING) or (random.random() > p):
        return text
    if not isinstance(keyword, str) or keyword.strip() == "":
        return text
    pattern = re.compile(rf"\b{re.escape(keyword)}\b", flags=re.IGNORECASE)
    return pattern.sub(mask_token, text)


## 5) Tokenisation + HuggingFace Datasets

In [58]:
from datasets import Dataset

def make_ds(df_in: pd.DataFrame, training: bool) -> Dataset:
    d = df_in[["text","keyword","binary"]].copy()
    if training:
        d["text"] = [
            maybe_mask_keyword(t, k, MASK_TOKEN, MASK_PROB)
            for t, k in zip(d["text"].tolist(), d["keyword"].tolist())
        ]
    ds = Dataset.from_pandas(d, preserve_index=False)

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

    ds = ds.map(tok, batched=True)
    ds = ds.rename_column("binary", "labels")
    ds = ds.remove_columns(["text","keyword"])
    ds.set_format("torch")
    return ds

ds_train = make_ds(train_tr_df, training=True)
ds_val   = make_ds(train_val_df, training=False)
ds_dev   = make_ds(dev_df, training=False)


Map: 100%|██████████| 2094/2094 [00:00<00:00, 25353.47 examples/s]


## 6) Class-weighted loss

In [59]:
pos = int(train_tr_df["binary"].sum())
neg = int(len(train_tr_df) - pos)
pos_weight = min(float(neg / max(pos, 1)), 10.0)
class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float)
print("pos:", pos, "neg:", neg, "pos_weight:", pos_weight)


pos: 675 neg: 6443 pos_weight: 9.545185185185185


## 7) Train (HuggingFace Trainer)

In [60]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1_pos": f1_score(labels, preds, pos_label=1),
        "precision_pos": precision_score(labels, preds, pos_label=1, zero_division=0),
        "recall_pos": recall_score(labels, preds, pos_label=1, zero_division=0),
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir="checkpoints/",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_pos",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    disable_tqdm=False,

    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    compute_metrics=compute_metrics,
)

trainer.train()

FINAL_DIR = "BestModel"
trainer.save_model(FINAL_DIR)            # saves model weights + config
tokenizer.save_pretrained(FINAL_DIR)     # saves tokenizer files


Epoch,Training Loss,Validation Loss,F1 Pos,Precision Pos,Recall Pos
1,0.438129,0.404921,0.452174,0.304985,0.873950
2,0.385748,0.398803,0.523659,0.419192,0.697479
3,0.356372,0.646959,0.572438,0.493902,0.680672
4,0.240966,0.928489,0.587302,0.556391,0.621849
5,0.200246,1.064078,0.595041,0.585366,0.605042


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.01it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

('BestModel/tokenizer_config.json', 'BestModel/tokenizer.json')

## 8) Threshold tuning on internal validation (improves F1)

In [61]:
from sklearn.metrics import f1_score

val_out = trainer.predict(ds_val)
val_probs = torch.softmax(torch.tensor(val_out.predictions), dim=1).numpy()[:, 1]
val_labels = val_out.label_ids

best_t, best_f1 = 0.5, -1
for t in np.linspace(0.30, 0.70, 9):
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(val_labels, preds, pos_label=1)
    if f1 > best_f1:
        best_f1, best_t = f1, float(t)

print("Best threshold:", best_t, "Best val F1:", best_f1)


Best threshold: 0.3 Best val F1: 0.596078431372549


## 9) Evaluate on official dev + write `dev.txt`

In [62]:
from sklearn.metrics import classification_report, f1_score

dev_out = trainer.predict(ds_dev)
dev_probs = torch.softmax(torch.tensor(dev_out.predictions), dim=1).numpy()[:, 1]
dev_labels = dev_out.label_ids
dev_preds = (dev_probs >= best_t).astype(int)

print("Official dev positive-class F1:", f1_score(dev_labels, dev_preds, pos_label=1))
print(classification_report(dev_labels, dev_preds, digits=4))

with open("dev.txt", "w") as f:
    for p in dev_preds:
        f.write(str(int(p)) + "\n")


Official dev positive-class F1: 0.5605700712589073
              precision    recall  f1-score   support

           0     0.9567    0.9451    0.9509      1895
           1     0.5315    0.5930    0.5606       199

    accuracy                         0.9117      2094
   macro avg     0.7441    0.7690    0.7557      2094
weighted avg     0.9163    0.9117    0.9138      2094



# INFERENCE CELLS
Run below to reproduce dev.txt and test.txt

## 10) Test predictions (when you upload official test inputs)
Uncomment and set `TEST_TSV_PATH` once you have the official test set (no labels).

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset

# Paths
DEV_TSV_PATH  = "dontpatronizeme_pcl.tsv"
TEST_TSV_PATH = "task4_test.tsv"

# Best threshold found on validation set
BEST_T = 0.7

FINAL_DIR = "BestModel"

from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR).to("cuda").eval()

def make_unlabeled_ds(df_in: pd.DataFrame) -> Dataset:
    d = df_in[["text"]].copy()
    d["text"] = d["text"].fillna("").astype(str)

    ds = Dataset.from_pandas(d, preserve_index=False)

    def tok(batch):
        texts = ["" if x is None else str(x) for x in batch["text"]]
        return tokenizer(texts, truncation=True, padding="max_length", max_length=MAX_LEN)

    ds = ds.map(tok, batched=True, load_from_cache_file=False)
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")
    return ds

def probs_from_trainer(ds: Dataset) -> np.ndarray:
    out = trainer.predict(ds)
    probs = torch.softmax(torch.tensor(out.predictions), dim=1).numpy()[:, 1]
    return probs

dev_ids = pd.read_csv("dev_semeval_parids-labels.csv")["par_id"].astype(int)

dev_full = pd.read_csv(
    DEV_TSV_PATH,
    sep="\t",
    skiprows=4,
    header=None,
    names=["par_id","art_id","keyword","country","text","label"],
)

dev_df = dev_full[dev_full["par_id"].isin(dev_ids)].copy()
ds_dev_infer = make_unlabeled_ds(dev_df)

dev_probs = probs_from_trainer(ds_dev_infer)
dev_preds = (dev_probs >= BEST_T).astype(int)

with open("dev.txt", "w") as f:
    for p in dev_preds:
        f.write(f"{int(p)}\n")

test_df = pd.read_csv(
    TEST_TSV_PATH,
    sep="\t",
    header=None,
    names=["par_id","art_id","keyword","country","text"],
)

ds_test_infer = make_unlabeled_ds(test_df)
test_probs = probs_from_trainer(ds_test_infer)
test_preds = (test_probs >= BEST_T).astype(int)

with open("test.txt", "w") as f:
    for p in test_preds:
        f.write(f"{int(p)}\n")


Loading weights:  42%|████▏     | 85/201 [00:00<00:00, 2961.96it/s, Materializing param=roberta.encoder.layer.4.intermediate.dense.weight]        

Map: 100%|██████████| 2094/2094 [00:00<00:00, 24137.19 examples/s]


Map: 100%|██████████| 3832/3832 [00:00<00:00, 24251.15 examples/s]
